# Phase 1 — Exploratory Data Analysis
## Online Retail II Dataset

This notebook walks through the full data ingestion and cleaning pipeline, then digs into the structure of the data before modelling begins.

**Sections**
1. Data loading & shape
2. Data quality & missing values
3. Revenue & quantity distributions
4. RFM analysis
5. Cohort retention
6. Category & geographic breakdown
7. Engineered feature distributions

## 1. Data Loading & Shape

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, calculate_rfm, engineer_features,
    build_cohort_matrix, build_master_customer_table
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df_raw = load_raw_data()
print(f'\nshape: {df_raw.shape}')
df_raw.head()

In [ ]:
print('column dtypes:')
print(df_raw.dtypes)
print('\nsample values per column:')
df_raw.describe(include='all').T

## 2. Data Quality & Missing Values

In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing_df[missing_df['missing_count'] > 0]['missing_pct'].plot(
    kind='barh', ax=ax, color='steelblue'
)
ax.set_title('Missing Value % by Column')
ax.set_xlabel('% missing')
plt.tight_layout()
plt.show()

In [ ]:
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
print('return rate distribution:')
print(return_features['return_rate'].describe().round(4))
print('\ncancellation rate distribution:')
print(cancel_features['cancellation_rate'].describe().round(4))

In [ ]:
df_customers, df_all = clean_data(df_raw)
print(f'\nrows retained (with customer_id): {len(df_customers):,}')
print(f'rows retained (all): {len(df_all):,}')

## 3. Revenue & Quantity Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

revenue_clip = df_customers['revenue'].clip(upper=df_customers['revenue'].quantile(0.99))
axes[0].hist(revenue_clip, bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Revenue per Line (clipped at 99th pct)')
axes[0].set_xlabel('Revenue (£)')

qty_clip = df_customers['quantity'].clip(upper=df_customers['quantity'].quantile(0.99))
axes[1].hist(qty_clip, bins=60, color='coral', edgecolor='white')
axes[1].set_title('Quantity per Line (clipped at 99th pct)')
axes[1].set_xlabel('Units')

plt.tight_layout()
plt.show()

In [ ]:
monthly = (
    df_customers
    .assign(month=df_customers['invoice_date'].dt.to_period('M'))
    .groupby('month')['revenue']
    .sum()
    .reset_index()
)
monthly['month'] = monthly['month'].astype(str)

fig = px.bar(monthly, x='month', y='revenue',
             title='Monthly Revenue', labels={'revenue': 'Revenue (£)', 'month': 'Month'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()